# Fine-tuning Laya on Wahy's quote findings (Kaggle, GPU T4 ×2)

Our version of the Laya repo's `notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb`. It trains **three variants
one after another** on the `laya_v1` dataset, evaluates each on the same test set, and compares them with base Laya
and the cached baselines. Walkthrough of every step: `docs/laya-finetune/kaggle-guide.md` in the Wahy repo.

| variant | labels | training data |
|---|---|---|
| `soft_asis` | soft (ε per source and question) | as is (~49 % "not a quote") |
| `soft_balanced` | soft | "not a quote" downsampled to 25 % |
| `hard_asis` | one-hot (ε = 0) | as is |

### Kaggle settings (right-hand panel)
* **Accelerator:** `GPU T4 x2` · **Internet:** `On` (needs a phone-verified account)
* **Input:** add the `laya_v1` dataset you uploaded (any slug: the notebook finds it by its `manifest.json`)
* **Unattended run:** *Save Version → Save & Run All (Commit)*. It runs in the background (up to 12 h) with the browser
  closed; everything lands in the version's **Output** tab (`/kaggle/working`). Expect ~1.5–2 h for the three variants.

## 1. Environment & dual T4 GPU check

In [ ]:
!nvidia-smi
import os, subprocess, torch

n_gpu = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | Visible GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")

assert n_gpu >= 2, (
    f"Expected 2 GPUs, but detected {n_gpu}! Notebook options -> Accelerator -> GPU T4 x2."
)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## 2. Install dependencies

In [ ]:
# CHANGED: pinned to the 0.4 line the Wahy data was validated with (laya.train / laya.calibrate are identical 0.4.12 -> 0.4.15)
!pip install -q -U "laya>=0.4.12,<0.5" "transformers>=4.48.0" safetensors huggingface_hub pandas matplotlib tabulate
import laya, transformers
print("Laya version        :", laya.__version__)
print("Transformers version:", transformers.__version__)
print("PyTorch version     :", torch.__version__)

## 3. Our dataset and the base checkpoint
**CHANGED vs the original:** the original reads `LocalLLaMA/typed-decisions` from the Hub and trains `convaiinnovations/laya`
— the *English* checkpoint (ModernBERT-large), which cannot read Arabic. We read our JSONL files and train
**`convaiinnovations/laya-multilingual`** (mmBERT-base), the checkpoint the Wahy app runs.

In [ ]:
import glob, json, sys
from pathlib import Path
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config

# the laya_v1 manifest (an upload may also hold other exports with their own manifest.json)
DATA = next(Path(p).parent for p in sorted(glob.glob("/kaggle/input/**/manifest.json", recursive=True))
            if json.loads(Path(p).read_text()).get("name") == "laya_v1")
manifest = json.loads((DATA / "manifest.json").read_text())
sys.path.insert(0, str(DATA))
import eval_laya

print("dataset:", DATA, "| created", manifest["created"], "| base model", manifest["base_model"])
for f, n in manifest["files"].items():
    print(f"  {f:28s} {n:5d} rows")
for v, d in manifest["variants"].items():
    print(f"  {v:14s} train rows {d['train_rows']}, 'not a quote' share {d['train_false_share']:.1%} - {d['about']}")

MODEL_ID = manifest["base_model"]          # CHANGED: convaiinnovations/laya-multilingual
model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)
print("base checkpoint:", model_dir, json.loads(Path(model_dir, "rl_agent_config.json").read_text())["encoder"])

## 4. Settings
The defaults are the original notebook's (effective batch 8 × 2 GPUs × 4 = 64, 4 epochs, RLCD). See the guide before
changing them. `VARIANTS` runs in this order; a failing variant is logged and the next one still runs.

In [ ]:
VARIANTS = ["soft_asis", "soft_balanced", "hard_asis"]
TRAIN_ARGS = "--epochs 4 --micro-batch 8 --grad-accum 4 --lr-encoder 2.5e-5 --lr-head 1e-4 --loss rlcd"
RUN_BASE_EVAL = True        # zero-shot base Laya on the same test set (the "before" line of the comparison)
APP_FALSE_SHARE = 0.10      # the app's real stream is ~10 % "not a quote" (used for the prior-corrected numbers)
PUSH_TO_HUB = False         # True: upload each model to a PRIVATE Hub repo (needs the HF_TOKEN secret)
HUB_PREFIX = "your-hf-username/laya-wahy"   # repo = f"{HUB_PREFIX}-{variant}"

WORK = Path("/kaggle/working")
(WORK / "results").mkdir(parents=True, exist_ok=True)
(WORK / "logs").mkdir(parents=True, exist_ok=True)

def run(cmd, log):
    """Run a shell command, stream its output here and into a log file; returns the exit code."""
    print("$", cmd, flush=True)
    with open(log, "w") as f:
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            f.write(line)
            if "Warning" not in line and "warn(" not in line:
                print(line, end="", flush=True)
        return p.wait()

## 5. Base Laya (zero-shot) on the test set

In [ ]:
if RUN_BASE_EVAL:
    rc = run(f'python {DATA}/eval_laya.py predict --model {model_dir} --name "base Laya (zero-shot)" '
             f'--test {DATA}/test.jsonl --out {WORK}/results/base.json --baselines {DATA}/baselines.json '
             f'--batch-size 32', WORK / "logs" / "eval_base.log")
    print("exit code", rc)

## 6. Train + evaluate the three variants
Per variant: `torchrun` launches `train_ddp.py` on both GPUs (DDP), then `eval_laya.py` scores the saved model on the test
set. **CHANGED vs the original:** calibration uses our `val.jsonl` (whole held-out lectures) instead of a random 10 % of
training items, and val loss / accuracy is printed after every epoch (the overfitting check).

In [ ]:
import time
status = {}
for v in VARIANTS:
    out = WORK / "models" / v
    t0 = time.time()
    try:
        rc = run(f"torchrun --standalone --nproc_per_node=2 {DATA}/train_ddp.py {model_dir} {out} "
                 f"--train {DATA}/{v}/train.jsonl --val {DATA}/{v}/val.jsonl --name laya-wahy-{v} {TRAIN_ARGS}",
                 WORK / "logs" / f"train_{v}.log")
        if rc != 0 or not (out / "model.safetensors").exists():
            status[v] = f"training failed (exit {rc}), see logs/train_{v}.log"
            continue
        share = manifest["variants"][v]["train_false_share"]
        rc = run(f"python {DATA}/eval_laya.py predict --model {out} --name {v} --test {DATA}/test.jsonl "
                 f"--out {WORK}/results/{v}.json --train-false-share {share} --app-false-share {APP_FALSE_SHARE} "
                 f"--batch-size 32", WORK / "logs" / f"eval_{v}.log")
        status[v] = f"ok in {(time.time() - t0) / 60:.0f} min" if rc == 0 else f"evaluation failed (exit {rc})"
    except Exception as e:
        status[v] = f"error: {e}"
    print(f"\n##### {v}: {status[v]}\n", flush=True)
json.dump(status, open(WORK / "results" / "status.json", "w"), indent=1)
status

## 7. Comparison
One table per test group. Rates are shown as `value [95 % interval]`: on your 198 cards (27 "not a quote") one mistake moves
the "not a quote" recall by ~4 points, so read the intervals before calling a difference real. Cached baselines (JEV / Gemini /
base Laya from the review notebook) only cover some test cards: the `cards` column says how many.

In [ ]:
from IPython.display import Markdown, display
files = sorted(glob.glob(str(WORK / "results" / "*.json")))
results = [json.load(open(f)) for f in files if not f.endswith("status.json")]
order = {"base Laya (zero-shot)": 0, **{v: i + 1 for i, v in enumerate(VARIANTS)}}
results.sort(key=lambda r: order.get(r["name"], 99))
report = []
for g in ["you", "you + Sonnet", "Sonnet · review lessons", "Sonnet · other lectures", "written text"]:
    t = f"### {g}\n" + eval_laya.compare(results, g)
    report.append(t)
    display(Markdown(t))
(WORK / "results" / "comparison.md").write_text("\n\n".join(report))

## 8. Training curves and calibration
Left: training loss (logged every 50 micro-batches on GPU 0). Right: val loss per epoch — if it rises while training loss
keeps falling, the model is memorising (fewer epochs). The fitted temperature > 1 means the model was over-confident on val
and is softened; < 1 sharpened.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for v in VARIANTS:
    p = WORK / "models" / v / "train_log.json"
    if not p.exists():
        continue
    h = json.load(open(p))
    spe = max(1, max(s["step"] for s in h["steps"])) if h["steps"] else 1
    ax[0].plot([(s["epoch"] - 1) + s["step"] / spe for s in h["steps"]], [s["loss"] for s in h["steps"]], label=v, alpha=.8)
    ep = [0] + [e["epoch"] for e in h["epochs"]]
    ax[1].plot(ep, [h["val_before"]["all"]["loss"]] + [e["val"]["all"]["loss"] for e in h["epochs"]], marker="o", label=v)
    cal = h.get("calibration", {})
    print(f"{v:14s} {h['seconds'] / 60:.0f} min | temperature (choice) {cal.get('temperature', ['?'])[0]} | "
          f"val accuracy {cal.get('val_at_fitted', {}).get('accuracy')} | ECE at T=1 {cal.get('val_at_t1', {}).get('ece')} "
          f"-> at fitted T {cal.get('val_at_fitted', {}).get('ece')} | issues {cal.get('report', {}).get('choice', {}).get('issues')}")
ax[0].set(title="training loss (RLCD)", xlabel="epoch"); ax[1].set(title="val soft cross-entropy", xlabel="epoch")
for a in ax: a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(WORK / "results" / "curves.png", dpi=120); plt.show()

## 9. Details on your labels: confusion matrix, threshold, span length, calibration
For each model: what it does with your 198 cards. *Prior-corrected* = the decision after shifting the probabilities from the
training class balance to the app's (~10 % "not a quote"). *Threshold sweep* = reject only when P(not a quote) ≥ t: a higher t
rejects fewer real quotes and catches fewer false findings.

In [ ]:
import pandas as pd
for r in results:
    g = r["groups"].get("you", {})
    v, b = g.get("verdict"), g.get("boundary")
    if not v:
        continue
    display(Markdown(f"#### {r['name']}"))
    display(pd.DataFrame(v["confusion (gold -> predicted)"]).T.rename_axis("gold \\ predicted"))
    pc = [k for k in v if k.startswith("prior-corrected")]
    if pc:
        print(pc[0], {k: eval_laya._fmt(x) if isinstance(x, list) else x for k, x in v[pc[0]].items()})
    display(pd.DataFrame([{k: eval_laya._fmt(x) if isinstance(x, list) else x for k, x in t.items()} for t in v["threshold sweep (reject if P(not a quote) >= t)"]]))
    display(pd.DataFrame([{k: eval_laya._fmt(x) if isinstance(x, list) else x for k, x in t.items()} for t in v["by span length"]]))
    display(pd.DataFrame(v["reliability"]))
    if b:
        print("boundary accuracy", eval_laya._fmt(b["accuracy"]))
        display(pd.DataFrame(b["confusion (gold -> predicted)"]).T)

## 10. (Optional) Push to the Hugging Face Hub — private
Add `HF_TOKEN` (a *write* token) under *Add-ons → Secrets*, set `PUSH_TO_HUB = True` and `HUB_PREFIX` in section 4.
Without it, download the models from the version's **Output** tab (`models/<variant>/`).

In [ ]:
if PUSH_TO_HUB:
    from huggingface_hub import HfApi
    from kaggle_secrets import UserSecretsClient
    api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
    for v in VARIANTS:
        out = WORK / "models" / v
        if not (out / "model.safetensors").exists():
            continue
        repo = f"{HUB_PREFIX}-{v}"
        api.create_repo(repo, repo_type="model", private=True, exist_ok=True)
        api.upload_folder(folder_path=str(out), repo_id=repo, repo_type="model",
                          commit_message=f"laya-multilingual fine-tuned on Wahy laya_v1 ({v})")
        print("pushed", repo)